# Transformación de IMFs a Grafo mediante Recurrence Plot

Este notebook carga los datos de las IMFs del MSCI World para su posterior transformación a grafo usando Recurrence Plot.


In [ ]:
import pandas as pd
import numpy as np
from typing import Optional

import torch
from torch_geometric.data import Data

from sklearn.metrics import mutual_info_score
from scipy.spatial import KDTree

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import networkx as nx


In [10]:
# Cargar datos de IMFs
archivo_imfs = "data/16dic25/msci_world_imfs.parquet"

df_imfs = pd.read_parquet(archivo_imfs, engine="pyarrow")
print(f"Datos cargados desde: {archivo_imfs}")
print(f"Shape del DataFrame: {df_imfs.shape}")
print(f"\nColumnas disponibles: {list(df_imfs.columns)}")
print(f"\nRango de fechas disponible:")
print(f"  - Desde: {df_imfs.index.min()}")
print(f"  - Hasta: {df_imfs.index.max()}")


Datos cargados desde: data/16dic25/msci_world_imfs.parquet
Shape del DataFrame: (3490, 11)

Columnas disponibles: ['IMF_1', 'IMF_2', 'IMF_3', 'IMF_4', 'IMF_5', 'IMF_6', 'IMF_7', 'IMF_8', 'IMF_9', 'IMF_10', 'Residuo']

Rango de fechas disponible:
  - Desde: 2012-01-12 00:00:00-05:00
  - Hasta: 2025-11-26 00:00:00-05:00


In [11]:
# Seleccionar IMF_1
imf_1 = np.asarray(df_imfs['IMF_1'].values)
print(f"Shape de IMF_1: {imf_1.shape}")
print(f"Primeros 10 valores de IMF_1: {imf_1[:10]}")


Shape de IMF_1: (3490,)
Primeros 10 valores de IMF_1: [-0.27589986 -0.2934861   0.08549549 -1.16096792  1.30101302 -0.84105598
  1.0079569  -0.56710679 -0.46208816  0.42473901]


In [ ]:
def calcular_informacion_mutual(serie: np.ndarray, tau_max: int = 50) -> tuple:
    """
    Calcula la información mutua entre la serie y su versión desplazada.
    
    Parameters
    ----------
    serie : np.ndarray
        Serie temporal unidimensional.
    tau_max : int, optional
        Máximo valor de tau a evaluar. Por defecto es 50.
    
    Returns
    -------
    tuple
        Tupla con (taus, valores_mi) donde taus son los valores de tau evaluados
        y valores_mi son los valores de información mutua correspondientes.
    """
    taus = np.arange(1, min(tau_max + 1, len(serie) // 2))
    valores_mi = []
    
    # Discretizar la serie para el cálculo de MI
    # Usar histograma para discretizar
    n_bins = int(np.sqrt(len(serie)))
    serie_discreta = np.digitize(serie, bins=np.linspace(serie.min(), serie.max(), n_bins))
    
    for tau in taus:
        serie_desplazada = np.roll(serie_discreta, -tau)
        # Asegurar que ambas series tengan la misma longitud
        serie_original = serie_discreta[:-tau] if tau > 0 else serie_discreta
        serie_desplazada = serie_desplazada[:-tau] if tau > 0 else serie_desplazada
        
        mi = mutual_info_score(serie_original, serie_desplazada)
        valores_mi.append(mi)
    
    return taus, np.array(valores_mi)


def seleccionar_tau(serie: np.ndarray, tau_max: int = 50) -> int:
    """
    Selecciona el valor óptimo de tau usando el primer mínimo de información mutua.
    
    Parameters
    ----------
    serie : np.ndarray
        Serie temporal unidimensional.
    tau_max : int, optional
        Máximo valor de tau a evaluar. Por defecto es 50.
    
    Returns
    -------
    int
        Valor óptimo de tau (delay).
    """
    taus, valores_mi = calcular_informacion_mutual(serie, tau_max)
    
    # Buscar el primer mínimo local
    # Si no hay mínimo claro, usar el primer valor donde la MI disminuye significativamente
    if len(valores_mi) > 2:
        # Buscar mínimos locales
        minimos = []
        for i in range(1, len(valores_mi) - 1):
            if valores_mi[i] < valores_mi[i-1] and valores_mi[i] < valores_mi[i+1]:
                minimos.append(i)
        
        if minimos:
            tau_optimo = taus[minimos[0]]
        else:
            # Si no hay mínimo claro, usar el primer valor donde la derivada cambia de signo
            derivada = np.diff(valores_mi)
            cambios = np.where(derivada > 0)[0]
            if len(cambios) > 0:
                tau_optimo = taus[cambios[0] + 1] if cambios[0] + 1 < len(taus) else taus[1]
            else:
                tau_optimo = taus[1]  # Valor por defecto
    else:
        tau_optimo = taus[0] if len(taus) > 0 else 1
    
    return tau_optimo


def construir_espacio_embedding(serie: np.ndarray, dim: int, tau: int) -> np.ndarray:
    """
    Construye el espacio de embedding usando el método de delay embedding.
    
    Parameters
    ----------
    serie : np.ndarray
        Serie temporal unidimensional.
    dim : int
        Dimensión del embedding.
    tau : int
        Delay (retraso) para el embedding.
    
    Returns
    -------
    np.ndarray
        Matriz de embedding de forma (N - (dim-1)*tau, dim) donde N es la longitud
        de la serie original.
    """
    n = len(serie)
    m = n - (dim - 1) * tau
    
    if m <= 0:
        raise ValueError(f"La combinación de dim={dim} y tau={tau} resulta en m={m} <= 0")
    
    embedding = np.zeros((m, dim))
    for i in range(dim):
        embedding[:, i] = serie[i * tau : i * tau + m]
    
    return embedding


def calcular_false_nearest_neighbors(serie: np.ndarray, tau: int, dim_max: int = 10, 
                                     umbral: float = 10.0, ratio_min: float = 0.1) -> int:
    """
    Calcula la dimensión óptima usando el método de False Nearest Neighbors (FNN).
    
    Parameters
    ----------
    serie : np.ndarray
        Serie temporal unidimensional.
    tau : int
        Delay (retraso) ya seleccionado.
    dim_max : int, optional
        Dimensión máxima a evaluar. Por defecto es 10.
    umbral : float, optional
        Umbral para considerar un vecino como "false". Por defecto es 10.0.
    ratio_min : float, optional
        Ratio mínimo de FNN para considerar que se ha alcanzado la dimensión óptima.
        Por defecto es 0.1 (10%).
    
    Returns
    -------
    int
        Dimensión óptima del embedding.
    """
    ratios_fnn = []
    
    for dim in range(1, dim_max + 1):
        try:
            # Construir embedding en dimensión dim
            embedding_dim = construir_espacio_embedding(serie, dim, tau)
            
            if dim == 1:
                ratios_fnn.append(1.0)  # En dim=1, todos son "false" por definición
                continue
            
            # Construir embedding en dimensión dim+1
            embedding_dim_plus = construir_espacio_embedding(serie, dim + 1, tau)
            
            # Construir KDTree para búsqueda de vecinos
            tree = KDTree(embedding_dim)
            
            # Para cada punto, encontrar el vecino más cercano
            n_puntos = len(embedding_dim)
            falsos_vecinos = 0
            
            for i in range(n_puntos):
                # Encontrar el vecino más cercano en dimensión dim
                # k=2 para obtener el punto mismo y su vecino más cercano
                distancias, indices = tree.query(embedding_dim[i], k=2)
                
                # Asegurar que sean arrays
                if not isinstance(distancias, np.ndarray):
                    distancias = np.array([distancias])
                if not isinstance(indices, np.ndarray):
                    indices = np.array([indices])
                
                # Encontrar el vecino más cercano (excluyendo el punto mismo)
                if len(distancias) > 1:
                    if indices[0] == i:
                        vecino_idx = int(indices[1])
                        dist_dim = float(distancias[1])
                    else:
                        vecino_idx = int(indices[0])
                        dist_dim = float(distancias[0])
                else:
                    continue
                
                # Calcular distancia en dimensión dim+1
                if vecino_idx < len(embedding_dim_plus) and i < len(embedding_dim_plus):
                    dist_dim_plus = np.linalg.norm(embedding_dim_plus[i] - embedding_dim_plus[vecino_idx])
                    
                    # Verificar si es un false neighbor
                    if dist_dim > 1e-10:  # Evitar división por cero
                        ratio = abs(embedding_dim_plus[i, -1] - embedding_dim_plus[vecino_idx, -1]) / dist_dim
                        if ratio > umbral:
                            falsos_vecinos += 1
            
            ratio_fnn = falsos_vecinos / n_puntos if n_puntos > 0 else 1.0
            ratios_fnn.append(ratio_fnn)
            
            # Si el ratio es muy bajo, probablemente hemos encontrado la dimensión óptima
            if ratio_fnn < ratio_min:
                return dim
                
        except Exception as e:
            print(f"Error al calcular FNN para dim={dim}: {e}")
            ratios_fnn.append(1.0)
    
    # Si no se encontró un mínimo claro, usar la dimensión con menor ratio
    if ratios_fnn:
        dim_optima = int(np.argmin(ratios_fnn)) + 1
        return dim_optima
    
    return 2  # Valor por defecto


def calcular_matriz_recurrencia(embedding: np.ndarray, umbral: Optional[float] = None, 
                                umbral_percentil: float = 10.0) -> np.ndarray:
    """
    Calcula la matriz de recurrencia a partir del espacio de embedding.
    
    Parameters
    ----------
    embedding : np.ndarray
        Matriz de embedding de forma (N, dim).
    umbral : float, optional
        Umbral absoluto para la distancia. Si es None, se usa umbral_percentil.
    umbral_percentil : float, optional
        Percentil para determinar el umbral de distancia. Por defecto es 10.0.
    
    Returns
    -------
    np.ndarray
        Matriz de recurrencia binaria de forma (N, N).
    """
    n = len(embedding)
    
    # Advertencia para matrices muy grandes
    if n > 5000:
        print(f"Advertencia: La matriz de recurrencia será muy grande ({n}x{n}). "
              f"Esto puede tardar varios minutos.")
    
    # Construir KDTree
    tree = KDTree(embedding)
    
    # Determinar umbral si no se proporciona
    if umbral is None:
        # Calcular umbral usando una muestra de distancias
        # Tomar una muestra aleatoria de puntos para estimar el percentil
        n_muestra = min(1000, n)
        indices_muestra = np.random.choice(n, size=n_muestra, replace=False)
        distancias_muestra = []
        
        for idx in indices_muestra:
            # Obtener distancias a los k vecinos más cercanos
            distancias, _ = tree.query(embedding[idx], k=min(100, n))
            distancias_muestra.extend(distancias[1:])  # Excluir distancia a sí mismo
        
        umbral = np.percentile(distancias_muestra, umbral_percentil)
        print(f"Umbral calculado (percentil {umbral_percentil}): {umbral:.4f}")
    
    # Construir matriz de recurrencia de manera eficiente
    # Inicializar matriz de ceros
    matriz_recurrencia = np.zeros((n, n), dtype=int)
    
    # Para cada punto, encontrar todos los puntos dentro del umbral
    for i in range(n):
        # Usar query_ball_point para encontrar todos los puntos dentro del radio
        indices_vecinos = tree.query_ball_point(embedding[i], r=umbral, p=2)
        # Convertir a array y eliminar el punto mismo
        indices_vecinos = np.array([idx for idx in indices_vecinos if idx != i])
        if len(indices_vecinos) > 0:
            matriz_recurrencia[i, indices_vecinos] = 1
    
    print(f"Número de recurrencias encontradas: {np.sum(matriz_recurrencia)}")
    
    return matriz_recurrencia


In [21]:
# Seleccionar tau (delay) usando información mutua
print("Seleccionando tau (delay) usando información mutua...")
tau_optimo = seleccionar_tau(imf_1, tau_max=50)
print(f"\nTau óptimo seleccionado: {tau_optimo}")


Seleccionando tau (delay) usando información mutua...

Tau óptimo seleccionado: 2


In [22]:
# Seleccionar dim usando False Nearest Neighbors
print("Seleccionando dim (dimensión de embedding) usando False Nearest Neighbors...")
dim_optima = calcular_false_nearest_neighbors(imf_1, tau=tau_optimo, dim_max=10)

print(f"\nDimensión óptima seleccionada: {dim_optima}")


Seleccionando dim (dimensión de embedding) usando False Nearest Neighbors...

Dimensión óptima seleccionada: 4


In [14]:
# Construir espacio de embedding
print(f"Construyendo espacio de embedding con dim={dim_optima} y tau={tau_optimo}...")
embedding = construir_espacio_embedding(imf_1, dim=dim_optima, tau=tau_optimo)

print(f"Shape del embedding: {embedding.shape}")
print(f"Primeras 5 filas del embedding:")
print(embedding[:5])


Construyendo espacio de embedding con dim=4 y tau=2...
Shape del embedding: (3484, 4)
Primeras 5 filas del embedding:
[[-0.27589986  0.08549549  1.30101302  1.0079569 ]
 [-0.2934861  -1.16096792 -0.84105598 -0.56710679]
 [ 0.08549549  1.30101302  1.0079569  -0.46208816]
 [-1.16096792 -0.84105598 -0.56710679  0.42473901]
 [ 1.30101302  1.0079569  -0.46208816 -0.61613414]]


In [18]:
# Calcular matriz de recurrencia
print("Calculando matriz de recurrencia...")
matriz_recurrencia = calcular_matriz_recurrencia(embedding, umbral_percentil=10.0)

print(f"Shape de la matriz de recurrencia: {matriz_recurrencia.shape}")
print(f"Número de recurrencias (enlaces): {np.sum(matriz_recurrencia)}")
print(f"Densidad del grafo: {np.sum(matriz_recurrencia) / (matriz_recurrencia.shape[0] * (matriz_recurrencia.shape[0] - 1)):.4f}")

# Tomar muestra de 500x500 para visualización
muestra_size = min(500, matriz_recurrencia.shape[0])
matriz_muestra = matriz_recurrencia[:muestra_size, :muestra_size]


Calculando matriz de recurrencia...
Umbral calculado (percentil 10.0): 0.3688
Número de recurrencias encontradas: 34492
Shape de la matriz de recurrencia: (3484, 3484)
Número de recurrencias (enlaces): 34492
Densidad del grafo: 0.0028


In [17]:
# Convertir matriz de recurrencia a formato edge_index para PyTorch Geometric
print("Convirtiendo matriz de recurrencia a formato de grafo...")

# Obtener índices de las aristas (donde matriz_recurrencia == 1)
edge_indices = np.where(matriz_recurrencia == 1)

# Crear edge_index en formato [2, num_edges]
edge_index = np.array([edge_indices[0], edge_indices[1]])
edge_index_torch = torch.tensor(edge_index, dtype=torch.long)

print(f"Número de nodos: {len(embedding)}")
print(f"Número de enlaces: {edge_index_torch.shape[1]}")

# Crear features de nodos usando los valores del embedding
# Usar el primer componente del embedding como feature (o todos los componentes)
node_features = torch.tensor(embedding[:, 0], dtype=torch.float).unsqueeze(1)

# Crear el objeto Data de PyTorch Geometric
data = Data(x=node_features, edge_index=edge_index_torch)

print(f"\nObjeto Data creado:")
print(f"  - Número de nodos: {data.num_nodes}")
print(f"  - Número de enlaces: {data.num_edges}")
print(f"  - Features de nodos shape: {data.x.shape}")
print(f"  - Edge index shape: {data.edge_index.shape}")
print(f"\nPrimeros 5 features de nodos:")
print(data.x[:5])
print(f"\nPrimeros 10 enlaces (edge_index):")
print(data.edge_index[:, :10])


Convirtiendo matriz de recurrencia a formato de grafo...
Número de nodos: 3484
Número de enlaces: 34802

Objeto Data creado:
  - Número de nodos: 3484
  - Número de enlaces: 34802
  - Features de nodos shape: torch.Size([3484, 1])
  - Edge index shape: torch.Size([2, 34802])

Primeros 5 features de nodos:
tensor([[-0.2759],
        [-0.2935],
        [ 0.0855],
        [-1.1610],
        [ 1.3010]])

Primeros 10 enlaces (edge_index):
tensor([[   0,    1,    1,    2,    2,    3,    3,    4,    5,    5],
        [ 366,  382, 2793, 2704, 2971,  338, 2729, 2706,   10,  852]])


In [19]:
# Filtrar el grafo para el rango 2020-2025

# Obtener una copia del índice para trabajar con él
indice_original = df_imfs.index.copy()

# Verificar y convertir el índice a datetime si es necesario
if not isinstance(indice_original, pd.DatetimeIndex):
    print("Convirtiendo índice a DatetimeIndex...")
    indice_original = pd.to_datetime(indice_original)

# Normalizar timezone si existe (convertir a naive datetime)
try:
    if hasattr(indice_original, 'tz') and indice_original.tz is not None:
        print("Normalizando timezone del índice...")
        # Convertir a UTC y luego quitar timezone
        indice_original = indice_original.tz_convert('UTC').tz_localize(None)
except Exception as e:
    print(f"Advertencia al normalizar timezone: {e}")
    # Intentar método alternativo
    try:
        indice_original = pd.to_datetime(indice_original).tz_localize(None)
    except:
        pass

# Definir rango de fechas (sin timezone para compatibilidad)
fecha_inicio = pd.Timestamp('2020-01-01')
fecha_fin = pd.Timestamp('2025-12-31')

# Normalizar fechas de búsqueda también si es necesario
if hasattr(fecha_inicio, 'tz') and fecha_inicio.tz is not None:
    fecha_inicio = fecha_inicio.tz_localize(None)
if hasattr(fecha_fin, 'tz') and fecha_fin.tz is not None:
    fecha_fin = fecha_fin.tz_localize(None)

# Mapear nodos del embedding a índices temporales originales
# El embedding tiene len(embedding) nodos que corresponden a los primeros len(embedding) puntos de la serie
# Por lo tanto, el nodo i del embedding corresponde al índice temporal i en la serie original
nodos_embedding = np.arange(len(embedding))
indices_temporales_embedding = nodos_embedding  # Mapeo directo

# Obtener máscara de fechas en el rango para los nodos del embedding
mascara_fechas = (indice_original[indices_temporales_embedding] >= fecha_inicio) & \
                 (indice_original[indices_temporales_embedding] <= fecha_fin)
indices_filtrados = np.where(mascara_fechas)[0]

print(f"Filtrando grafo para el rango {fecha_inicio.date()} a {fecha_fin.date()}")
print(f"Nodos en el rango: {len(indices_filtrados)}")
if len(indices_filtrados) > 0:
    print(f"Índices del embedding: {indices_filtrados[0]} a {indices_filtrados[-1]}")
    print(f"Índices temporales originales: {indices_temporales_embedding[indices_filtrados[0]]} a {indices_temporales_embedding[indices_filtrados[-1]]}")
else:
    print("⚠ No se encontraron nodos en el rango especificado")
    if len(indices_temporales_embedding) > 0:
        fecha_min_embedding = indice_original[indices_temporales_embedding[0]]
        fecha_max_embedding = indice_original[indices_temporales_embedding[-1]]
        print(f"Rango disponible: {fecha_min_embedding} a {fecha_max_embedding}")

# Crear un conjunto de nodos válidos para el filtro
nodos_validos = set(indices_filtrados)

# Filtrar edge_index para mantener solo enlaces entre nodos válidos
edge_index_filtrado = data.edge_index.cpu().numpy()
enlaces_filtrados = []
for i in range(edge_index_filtrado.shape[1]):
    src = int(edge_index_filtrado[0, i])
    dst = int(edge_index_filtrado[1, i])
    if src in nodos_validos and dst in nodos_validos:
        enlaces_filtrados.append([src, dst])

if len(enlaces_filtrados) > 0:
    enlaces_filtrados = np.array(enlaces_filtrados, dtype=np.int64)
else:
    enlaces_filtrados = np.array([], dtype=np.int64).reshape(0, 2)
print(f"Enlaces en el subgrafo: {len(enlaces_filtrados)}")

# Crear mapeo de índices originales a índices locales (0, 1, 2, ...)
mapeo_indices = {int(idx_original): int(idx_local) for idx_local, idx_original in enumerate(indices_filtrados)}
if len(enlaces_filtrados) > 0:
    enlaces_mapeados = np.array([[mapeo_indices[int(src)], mapeo_indices[int(dst)]] 
                                  for src, dst in enlaces_filtrados], dtype=np.int64)
else:
    enlaces_mapeados = np.array([], dtype=np.int64).reshape(0, 2)

# Obtener valores de IMF_1 y fechas para los nodos filtrados
# Los nodos del embedding corresponden a índices temporales en la serie original
indices_temporales_filtrados = indices_temporales_embedding[indices_filtrados]
valores_imf_filtrados = imf_1[indices_temporales_filtrados]
fechas_filtradas = indice_original[indices_temporales_filtrados]

print(f"\nDatos del subgrafo:")
print(f"  - Nodos: {len(indices_filtrados)}")
print(f"  - Enlaces: {len(enlaces_mapeados)}")
if len(fechas_filtradas) > 0:
    fecha_min = fechas_filtradas[0]
    fecha_max = fechas_filtradas[-1]
    if isinstance(fecha_min, pd.Timestamp):
        print(f"  - Rango de fechas: {fecha_min.date()} a {fecha_max.date()}")
    else:
        print(f"  - Rango de fechas: {fecha_min} a {fecha_max}")


Normalizando timezone del índice...
Filtrando grafo para el rango 2020-01-01 a 2025-12-31
Nodos en el rango: 1479
Índices del embedding: 2005 a 3483
Índices temporales originales: 2005 a 3483
Enlaces en el subgrafo: 4178

Datos del subgrafo:
  - Nodos: 1479
  - Enlaces: 4178
  - Rango de fechas: 2020-01-02 a 2025-11-18


In [20]:
# Crear visualización del grafo con Plotly usando layout de fuerza
# Incluyendo también la serie temporal de la IMF en la misma figura

# Validar que hay datos para visualizar
if len(indices_filtrados) == 0:
    raise ValueError("No hay nodos en el rango de fechas especificado. Ajusta el rango de fechas.")

if len(enlaces_mapeados) == 0:
    print("⚠ Advertencia: No hay enlaces en el subgrafo filtrado.")

# Convertir valores a arrays de numpy para asegurar compatibilidad
valores_imf_array = np.array(valores_imf_filtrados, dtype=np.float64)

# Crear grafo NetworkX para calcular layout
print("Creando grafo NetworkX...")
G = nx.Graph()
G.add_nodes_from(range(len(indices_filtrados)))
if len(enlaces_mapeados) > 0:
    # Asegurar que los enlaces sean tuplas de enteros
    enlaces_tuplas = [(int(edge[0]), int(edge[1])) for edge in enlaces_mapeados]
    G.add_edges_from(enlaces_tuplas)
    print(f"Grafo creado con {G.number_of_nodes()} nodos y {G.number_of_edges()} enlaces")

# Calcular layout usando spring layout (fuerza dirigida)
print("Calculando layout del grafo...")
pos = nx.spring_layout(G, k=1, iterations=50, seed=42)

# Extraer posiciones
x_pos = np.array([pos[i][0] for i in range(len(indices_filtrados))])
y_pos = np.array([pos[i][1] for i in range(len(indices_filtrados))])

# Preparar texto de hover de forma más eficiente
def formatear_fecha(fecha):
    """Formatea una fecha de forma segura."""
    try:
        if isinstance(fecha, pd.Timestamp):
            return fecha.strftime('%Y-%m-%d')
        elif isinstance(fecha, str):
            return fecha
        else:
            return str(pd.Timestamp(fecha).strftime('%Y-%m-%d'))
    except:
        return str(fecha)

textos_hover = [
    f"Fecha: {formatear_fecha(fecha)}<br>Valor IMF_1: {val:.4f}<br>Índice: {idx}"
    for fecha, val, idx in zip(fechas_filtradas, valores_imf_array, indices_temporales_filtrados)
]

# Crear trazas para los enlaces de forma más eficiente
print("Creando trazas de enlaces...")
edge_x = []
edge_y = []
if len(enlaces_mapeados) > 0:
    for edge in enlaces_mapeados:
        src_idx = int(edge[0])
        dst_idx = int(edge[1])
        x0, y0 = pos[src_idx]
        x1, y1 = pos[dst_idx]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

# Crear traza de enlaces
edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    showlegend=False
)

# Calcular rango de valores para la escala de colores (compartida entre grafo y serie temporal)
valor_min = np.min(valores_imf_array)
valor_max = np.max(valores_imf_array)

# Crear traza para los nodos
print("Creando traza de nodos...")
node_trace = go.Scatter(
    x=x_pos,
    y=y_pos,
    mode='markers',
    marker=dict(
        size=8,
        color=valores_imf_array,
        colorscale='Viridis',
        cmin=valor_min,  # Rango mínimo para la escala
        cmax=valor_max,  # Rango máximo para la escala
        showscale=False,  # Ocultar colorbar del grafo, se mostrará solo en la serie temporal
        line=dict(width=1, color='black')
    ),
    text=textos_hover,
    hoverinfo='text',
    showlegend=False
)

# Crear traza para la serie temporal con colores basados en valores (mismo heatmap que el grafo)
print("Creando traza de serie temporal...")
serie_temporal_trace = go.Scatter(
    x=fechas_filtradas,
    y=valores_imf_array,
    mode='lines+markers',
    name='IMF_1',
    line=dict(color='rgba(128,128,128,0.3)', width=1),  # Línea sutil gris para conectar puntos
    marker=dict(
        size=4,
        color=valores_imf_array,
        colorscale='Viridis',
        cmin=valor_min,  # Mismo rango que el grafo
        cmax=valor_max,  # Mismo rango que el grafo
        showscale=True,  # Solo esta traza mostrará la colorbar
        colorbar=dict(title="Valor IMF_1", x=1.02, len=0.4, y=0.25),
        line=dict(width=0.5, color='white')
    ),
    hovertemplate='Fecha: %{x}<br>Valor IMF_1: %{y:.4f}<extra></extra>'
)

# Crear figura con subplots: grafo arriba, serie temporal abajo
print("Creando figura con subplots...")
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.6, 0.4],  # 60% para el grafo, 40% para la serie temporal
    subplot_titles=(
        f'Grafo Recurrencia - MSCI World IMF_1 ({fecha_inicio.date()} a {fecha_fin.date()})',
        'Serie Temporal IMF_1'
    ),
    vertical_spacing=0.12
)

# Añadir trazas del grafo al primer subplot
fig.add_trace(edge_trace, row=1, col=1)
fig.add_trace(node_trace, row=1, col=1)

# Añadir traza de serie temporal al segundo subplot
fig.add_trace(serie_temporal_trace, row=2, col=1)

# Actualizar layout del primer subplot (grafo)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    row=1, col=1
)
fig.update_yaxes(
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    row=1, col=1
)

# Actualizar layout del segundo subplot (serie temporal)
fig.update_xaxes(
    title_text="Fecha",
    row=2, col=1
)
fig.update_yaxes(
    title_text="Valor IMF_1",
    row=2, col=1
)

# Actualizar layout general
fig.update_layout(
    title=dict(
        text=f'Grafo Recurrencia y Serie Temporal - MSCI World IMF_1 ({fecha_inicio.date()} a {fecha_fin.date()})',
        x=0.5,
        xanchor='center',
        font=dict(size=18)
    ),
    showlegend=False,
    hovermode='closest',
    height=900,
    margin=dict(b=50, l=50, r=50, t=80),
    annotations=[
        dict(
            text=f"Nodos: {len(indices_filtrados)} | Enlaces: {len(enlaces_mapeados)}",
            showarrow=False,
            xref="paper", yref="paper",
            x=0.005, y=0.48,
            xanchor='left', yanchor='bottom',
            font=dict(size=12)
        )
    ],
    plot_bgcolor='white'
)

# Guardar en HTML
archivo_html = "data/16dic25/grafo_recurrencia_2020_2025.html"
print(f"Guardando visualización en {archivo_html}...")
fig.write_html(archivo_html)
print(f"✓ Visualización guardada exitosamente")

# Mostrar la figura
fig.show()


Creando grafo NetworkX...
Grafo creado con 1479 nodos y 2089 enlaces
Calculando layout del grafo...
Creando trazas de enlaces...
Creando traza de nodos...
Creando traza de serie temporal...
Creando figura con subplots...
Guardando visualización en data/16dic25/grafo_recurrencia_2020_2025.html...
✓ Visualización guardada exitosamente
